# Notebook 07: Benchmarking and Candidate Ranking

## Objective
Transform ATS scores, semantic match scores, and skill gap outputs into
benchmarked candidate signals with domain-stratified percentile ranks.
Produce the final ranked dataset for Streamlit application consumption.

## Methodology
1. Load and validate all input artifacts
2. Compute domain-stratified percentile ranks for three scores
3. Apply low-confidence flags for thin-population domains
4. Assign experience band labels
5. Export benchmark dataset and supporting metadata

## Assumptions
- Domain-level percentile stratification is the appropriate unit of comparison
- Management and Engineering populations are below the reliability threshold
- Semantic display scores are domain-relative signals, not absolute quality measures
- ATS scores are finalized and consumed as-is
- Experience band is a display label only, not a ranking dimension

## Inputs
- outputs/ats_scored_candidates.csv
- outputs/semantic_match_scores.csv
- outputs/skill_gap_outputs.csv

## Outputs
- outputs/ats_benchmark_scores.csv
- outputs/benchmark_metadata.json

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from scipy import stats

OUTPUT_DIR = Path("../outputs")

# loading input artifacts
ats = pd.read_csv(OUTPUT_DIR / "ats_scored_candidates.csv")
semantic = pd.read_csv(OUTPUT_DIR / "semantic_match_scores.csv")
gap = pd.read_csv(OUTPUT_DIR / "skill_gap_outputs.csv")

# validating shapes
print("ats shape:      ", ats.shape)
print("semantic shape: ", semantic.shape)
print("gap shape:      ", gap.shape)
print()

# validating required columns
required_ats = [
    "candidate_id", "primary_domain", "years_experience",
    "experience_band", "ats_score",
    "score_education", "score_experience", "score_skills",
    "score_flags", "score_completeness"
]
required_semantic = [
    "candidate_id", "primary_domain",
    "top_job_similarity", "display_score"
]
required_gap = [
    "candidate_id", "primary_domain",
    "missing_skills", "gap_count", "domain_coverage_pct"
]

for col in required_ats:
    assert col in ats.columns, f"Missing in ats: {col}"
for col in required_semantic:
    assert col in semantic.columns, f"Missing in semantic: {col}"
for col in required_gap:
    assert col in gap.columns, f"Missing in gap: {col}"

print("All required columns present.")
print()

# checking candidate_id consistency across all three artifacts
ats_ids = set(ats["candidate_id"])
sem_ids = set(semantic["candidate_id"])
gap_ids = set(gap["candidate_id"])

print("Candidate IDs in ats not in semantic: ", len(ats_ids - sem_ids))
print("Candidate IDs in ats not in gap:      ", len(ats_ids - gap_ids))
print("Candidate IDs in semantic not in ats: ", len(sem_ids - ats_ids))
print("Candidate IDs in gap not in ats:      ", len(gap_ids - ats_ids))
print()

# checking domain consistency
print("Domain distribution — ats:")
print(ats["primary_domain"].value_counts().to_string())
print()
print("Domain distribution — semantic:")
print(semantic["primary_domain"].value_counts().to_string())
print()
print("Domain distribution — gap:")
print(gap["primary_domain"].value_counts().to_string())
print()

# checking for nulls in scoring columns
print("Null counts — ats score columns:")
score_cols = ["ats_score", "score_education", "score_experience",
              "score_skills", "score_flags", "score_completeness"]
print(ats[score_cols].isnull().sum().to_string())
print()
print("Null counts — semantic:")
print(semantic[["top_job_similarity", "display_score"]].isnull().sum().to_string())
print()
print("Null counts — gap:")
print(gap[["gap_count", "domain_coverage_pct"]].isnull().sum().to_string())


ats shape:       (5000, 13)
semantic shape:  (5000, 6)
gap shape:       (5000, 5)

All required columns present.

Candidate IDs in ats not in semantic:  0
Candidate IDs in ats not in gap:       0
Candidate IDs in semantic not in ats:  0
Candidate IDs in gap not in ats:       0

Domain distribution — ats:
primary_domain
IT              2772
Data Science    1002
HR               501
Legal            397
Engineering      189
Management       139

Domain distribution — semantic:
primary_domain
IT              2772
Data Science    1002
HR               501
Legal            397
Engineering      189
Management       139

Domain distribution — gap:
primary_domain
IT              2772
Data Science    1002
HR               501
Legal            397
Engineering      189
Management       139

Null counts — ats score columns:
ats_score             0
score_education       0
score_experience      0
score_skills          0
score_flags           0
score_completeness    0

Null counts — semantic:
top_job

## Section 1: Artifact Merge and Pre-Benchmarking Distribution Check

Merging all three input artifacts on candidate_id into a single working
dataset. Inspecting score distributions per domain before computing
percentile ranks to confirm that domain-stratified ranking is the correct
approach and to surface any unexpected artifacts.

In [ ]:
# merging all three artifacts on candidate_id
df = ats.merge(
    semantic[["candidate_id", "top_job_similarity", "display_score"]],
    on="candidate_id",
    how="left"
).merge(
    gap[["candidate_id", "gap_count", "domain_coverage_pct"]],
    on="candidate_id",
    how="left"
)

print("Merged dataset shape:", df.shape)
print("Null check after merge:")
check_cols = ["ats_score", "display_score", "domain_coverage_pct"]
print(df[check_cols].isnull().sum().to_string())
print()

# per-domain score distributions for the three scores receiving percentile ranks
domains = ["IT", "Data Science", "HR", "Legal", "Engineering", "Management"]
score_targets = {
    "ats_score":            "ATS Score",
    "display_score":        "Semantic Display Score",
    "domain_coverage_pct":  "Domain Coverage Pct"
}

for col, label in score_targets.items():
    print(f" {label} by domain ")
    summary = (
        df.groupby("primary_domain")[col]
        .agg(["count", "mean", "std", "min", "max"])
        .round(2)
        .loc[domains]
    )
    print(summary.to_string())
    print()

Merged dataset shape: (5000, 17)
Null check after merge:
ats_score              0
display_score          0
domain_coverage_pct    0

--- ATS Score by domain ---
                count   mean    std    min    max
primary_domain                                   
IT               2772  68.62  10.48  36.90  89.96
Data Science     1002  67.08   9.99  38.24  84.65
HR                501  72.23  11.01  43.17  89.08
Legal             397  70.28  10.09  43.24  84.65
Engineering       189  68.28   9.70  43.11  84.53
Management        139  78.86  10.82  43.24  89.96

--- Semantic Display Score by domain ---
                count   mean    std    min     max
primary_domain                                    
IT               2772  51.92  12.44   8.14   81.24
Data Science     1002  30.85   9.39   5.17   56.61
HR                501  43.27   7.22  29.38   61.46
Legal             397  74.82  18.22  50.54  100.00
Engineering       189  48.80   9.52  33.34   61.46
Management        139  11.58   9.80   0.

## Section 2: Domain-Stratified Percentile Ranks

Percentile ranks are computed independently for each domain using
scipy.stats.percentileofscore with kind='rank'. This handles ties
consistently and is stable on small populations.

Three scores receive domain percentile ranks:
- ats_score
- display_score (semantic match)
- domain_coverage_pct

Gap count is excluded as it is directionally redundant with domain coverage.
Experience band is retained as a display label only.

In [3]:
from scipy.stats import percentileofscore

def compute_domain_percentiles(df, score_col, output_col):
    percentiles = []
    for domain in df["primary_domain"].unique():
        mask = df["primary_domain"] == domain
        domain_scores = df.loc[mask, score_col].values
        domain_pcts = [
            round(percentileofscore(domain_scores, s, kind="rank"), 2)
            for s in domain_scores
        ]
        idx = df.index[mask]
        percentiles.extend(zip(idx, domain_pcts))

    pct_series = pd.Series(
        {idx: pct for idx, pct in percentiles},
        name=output_col
    )
    return pct_series

# computing percentile ranks for all three scores
df["ats_percentile_domain"] = compute_domain_percentiles(
    df, "ats_score", "ats_percentile_domain"
)
df["semantic_percentile_domain"] = compute_domain_percentiles(
    df, "display_score", "semantic_percentile_domain"
)
df["coverage_percentile_domain"] = compute_domain_percentiles(
    df, "domain_coverage_pct", "coverage_percentile_domain"
)

# validating distributions per domain
print("ATS percentile — mean per domain (expect ~50):")
print(df.groupby("primary_domain")["ats_percentile_domain"]
      .agg(["mean", "min", "max"]).round(2).loc[domains].to_string())
print()

print("Semantic percentile — mean per domain (expect ~50):")
print(df.groupby("primary_domain")["semantic_percentile_domain"]
      .agg(["mean", "min", "max"]).round(2).loc[domains].to_string())
print()

print("Coverage percentile — mean per domain (expect ~50):")
print(df.groupby("primary_domain")["coverage_percentile_domain"]
      .agg(["mean", "min", "max"]).round(2).loc[domains].to_string())
print()

# checking for nulls
pct_cols = [
    "ats_percentile_domain",
    "semantic_percentile_domain",
    "coverage_percentile_domain"
]
print("Null counts in percentile columns:")
print(df[pct_cols].isnull().sum().to_string())

ATS percentile — mean per domain (expect ~50):
                 mean   min     max
primary_domain                     
IT              50.02  0.05  100.00
Data Science    50.05  0.10   99.40
HR              50.10  0.40   99.90
Legal           50.13  0.38   96.22
Engineering     50.27  0.53   97.88
Management      50.36  0.72   99.28

Semantic percentile — mean per domain (expect ~50):
                 mean   min     max
primary_domain                     
IT              50.02  0.05   99.96
Data Science    50.05  0.15  100.00
HR              50.10  0.50   99.60
Legal           50.13  3.27   96.85
Engineering     50.26  3.70   97.35
Management      50.36  4.68   98.20

Coverage percentile — mean per domain (expect ~50):
                 mean    min    max
primary_domain                     
IT              50.02  14.70  92.98
Data Science    50.05   2.99  79.14
HR              50.10  40.82  90.82
Legal           50.12   8.82  90.55
Engineering     50.27   3.44  79.10
Management      50.

## Section 3: Confidence Flags and Experience Band

Domains with fewer than 200 records produce unstable percentile estimates.
Management (139) and Engineering (189) are flagged with low_confidence_flag = True.

Experience band is assigned as a display label for the Streamlit application.
It is not used as a ranking dimension. The 11-20 year range is binned into
a single Expert bucket consistent with prior notebooks.

In [4]:
# low confidence threshold
LOW_CONFIDENCE_THRESHOLD = 200

domain_population = df["primary_domain"].value_counts().to_dict()

df["low_confidence_flag"] = df["primary_domain"].apply(
    lambda d: domain_population[d] < LOW_CONFIDENCE_THRESHOLD
)

print("Low confidence flag distribution:")
print(df.groupby("primary_domain")["low_confidence_flag"]
      .agg(["first", "count"])
      .rename(columns={"first": "low_confidence", "count": "n_candidates"})
      .loc[domains].to_string())
print()

# experience band assignment
# consistent with bins used in Notebook 05
bins   = [0, 3, 6, 10, 20]
labels = ["Junior (1-3)", "Mid (4-6)", "Senior (7-10)", "Expert (11-20)"]

# experience_band already exists in ats artifact from Notebook 05
# confirming it is present and correctly formed before using it
if "experience_band" in df.columns:
    print("experience_band already present in artifact.")
    print("Value counts:")
    print(df["experience_band"].value_counts().to_string())
    print()
    print("Null count:", df["experience_band"].isnull().sum())
else:
    print("experience_band not found — recomputing.")
    df["experience_band"] = pd.cut(
        df["years_experience"],
        bins=bins,
        labels=labels
    )
    print("experience_band computed.")
    print(df["experience_band"].value_counts().to_string())

Low confidence flag distribution:
                low_confidence  n_candidates
primary_domain                              
IT                       False          2772
Data Science             False          1002
HR                       False           501
Legal                    False           397
Engineering               True           189
Management                True           139

experience_band already present in artifact.
Value counts:
experience_band
Senior (7-10)     1821
Mid (4-6)         1475
Junior (1-3)      1171
Expert (11-20)     533

Null count: 0


## Section 4: Final Dataset Assembly and Export

Selecting the output columns for the benchmark artifact consumed by
the Streamlit application. Exporting the ranked dataset and a supporting
metadata file that documents population sizes, score ranges, confidence
rules, and known limitations.

The metadata file decouples calibration parameters from UI logic.
The Streamlit application reads this file to render appropriate caveats
without hardcoding domain-specific knowledge in the interface layer.

In [5]:
# assembling final output columns
output_cols = [
    "candidate_id",
    "primary_domain",
    "primary_role",
    "years_experience",
    "experience_band",
    "highest_education",
    "institution_tier",
    "ats_score",
    "score_education",
    "score_experience",
    "score_skills",
    "score_flags",
    "score_completeness",
    "ats_percentile_domain",
    "display_score",
    "semantic_percentile_domain",
    "domain_coverage_pct",
    "coverage_percentile_domain",
    "gap_count",
    "low_confidence_flag"
]

benchmark = df[output_cols].copy()

print("Benchmark dataset shape:", benchmark.shape)
print()
print("Null check:")
print(benchmark.isnull().sum()[benchmark.isnull().sum() > 0])
print("No nulls." if benchmark.isnull().sum().sum() == 0 else "Nulls found.")
print()

# exporting benchmark dataset
benchmark.to_csv(OUTPUT_DIR / "ats_benchmark_scores.csv", index=False)
size_kb = round(
    (OUTPUT_DIR / "ats_benchmark_scores.csv").stat().st_size / 1024, 1
)
print(f"ats_benchmark_scores.csv exported: {benchmark.shape}  {size_kb} KB")
print()

# building benchmark metadata
domain_stats = {}
for domain in domains:
    mask = df["primary_domain"] == domain
    domain_df = df[mask]
    domain_stats[domain] = {
        "n_candidates":               int(domain_population[domain]),
        "low_confidence":             bool(domain_population[domain] < LOW_CONFIDENCE_THRESHOLD),
        "ats_score_mean":             round(float(domain_df["ats_score"].mean()), 2),
        "ats_score_std":              round(float(domain_df["ats_score"].std()), 2),
        "ats_score_min":              round(float(domain_df["ats_score"].min()), 2),
        "ats_score_max":              round(float(domain_df["ats_score"].max()), 2),
        "semantic_display_mean":      round(float(domain_df["display_score"].mean()), 2),
        "semantic_display_std":       round(float(domain_df["display_score"].std()), 2),
        "coverage_pct_mean":          round(float(domain_df["domain_coverage_pct"].mean()), 2),
        "coverage_pct_std":           round(float(domain_df["domain_coverage_pct"].std()), 2)
    }

metadata = {
    "benchmarking_method":        "domain_stratified_percentile",
    "percentile_function":        "scipy.stats.percentileofscore",
    "percentile_kind":            "rank",
    "low_confidence_threshold":   LOW_CONFIDENCE_THRESHOLD,
    "scores_ranked": [
        "ats_score",
        "display_score",
        "domain_coverage_pct"
    ],
    "gap_count_ranked":           False,
    "experience_band_ranked":     False,
    "domain_stats":               domain_stats,
    "semantic_score_caveat":      (
        "Semantic display scores are not cross-domain comparable. "
        "Domain means range from 11.58 (Management) to 74.82 (Legal) "
        "due to vocabulary alignment differences in the synthetic corpus, "
        "not candidate quality differences. Use domain percentile rank only."
    ),
    "coverage_caveat":            (
        "Domain coverage percentile is compressed for HR and Management "
        "due to narrow raw coverage ranges. Use as a weak relative signal only."
    ),
    "synthetic_data_limitations": (
        "All scores are derived from synthetic candidate profiles. "
        "ATS completeness component carries zero variance. "
        "Semantic scores are compressed due to skill sentence length asymmetry. "
        "Coverage percentiles reflect 5-6 token profiles against 10-22 token pools. "
        "Recalibrate all thresholds and rescaling parameters on real resume data."
    ),
    "source_notebooks": {
        "ats_scoring":       "Notebook 05",
        "semantic_matching": "Notebook 06",
        "skill_gap":         "Notebook 06",
        "benchmarking":      "Notebook 07"
    }
}

with open(OUTPUT_DIR / "benchmark_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

meta_size_kb = round(
    (OUTPUT_DIR / "benchmark_metadata.json").stat().st_size / 1024, 1
)
print(f"benchmark_metadata.json exported: {meta_size_kb} KB")
print()

# confirming all outputs directory contents
print("outputs/ directory contents:")
for f in sorted(OUTPUT_DIR.glob("*")):
    size_kb = round(f.stat().st_size / 1024, 1)
    print(f"  {f.name:<55} {size_kb} KB")

Benchmark dataset shape: (5000, 20)

Null check:
Series([], dtype: int64)
No nulls.

ats_benchmark_scores.csv exported: (5000, 20)  664.4 KB

benchmark_metadata.json exported: 3.4 KB

outputs/ directory contents:
  ats_benchmark_scores.csv                                664.4 KB
  ats_scored_candidates.csv                               479.4 KB
  ats_scoring_artifacts.json                              3.5 KB
  benchmark_metadata.json                                 3.4 KB
  candidate_skill_profiles.csv                            292.4 KB
  cleaned_skill_vocabulary.csv                            4.8 KB
  curated_job_descriptions.csv                            2381.1 KB
  domain_distribution_summary.csv                         0.3 KB
  embedding_artifacts.pkl                                 7961.3 KB
  esco_skill_mapping.csv                                  2.7 KB
  job_title_taxonomy.csv                                  15.6 KB
  normalized_candidate_skill_profiles.csv                 7

## Section 5: Validation

Reloading exported artifacts and confirming internal consistency.
Spot checking percentile distributions and flag assignments on the
reloaded data before closing the notebook.

In [6]:
# reloading exported artifacts
benchmark_check = pd.read_csv(OUTPUT_DIR / "ats_benchmark_scores.csv")
with open(OUTPUT_DIR / "benchmark_metadata.json") as f:
    metadata_check = json.load(f)

print("Reloaded benchmark shape:", benchmark_check.shape)
print("Reloaded metadata keys:", list(metadata_check.keys()))
print()

# confirming low confidence flag counts
print("Low confidence flag counts:")
print(benchmark_check["low_confidence_flag"].value_counts().to_string())
print()

# confirming percentile ranges per domain for all three ranked scores
pct_cols = {
    "ats_percentile_domain":        "ATS",
    "semantic_percentile_domain":   "Semantic",
    "coverage_percentile_domain":   "Coverage"
}

for col, label in pct_cols.items():
    print(f"{label} percentile — min/mean/max per domain:")
    summary = (
        benchmark_check.groupby("primary_domain")[col]
        .agg(["min", "mean", "max"])
        .round(2)
        .loc[domains]
    )
    print(summary.to_string())
    print()

# confirming experience band distribution
print("Experience band distribution:")
print(benchmark_check["experience_band"].value_counts().to_string())
print()

# spot checking 3 candidates across different domains
print("Spot check — one candidate per domain (first record):")
spot_cols = [
    "candidate_id", "primary_domain", "years_experience",
    "experience_band", "ats_score", "ats_percentile_domain",
    "display_score", "semantic_percentile_domain",
    "domain_coverage_pct", "coverage_percentile_domain",
    "low_confidence_flag"
]
for domain in domains:
    row = benchmark_check[benchmark_check["primary_domain"] == domain].iloc[0]
    print(f"\n  {domain}")
    for col in spot_cols:
        print(f"    {col:<30} {row[col]}")

Reloaded benchmark shape: (5000, 20)
Reloaded metadata keys: ['benchmarking_method', 'percentile_function', 'percentile_kind', 'low_confidence_threshold', 'scores_ranked', 'gap_count_ranked', 'experience_band_ranked', 'domain_stats', 'semantic_score_caveat', 'coverage_caveat', 'synthetic_data_limitations', 'source_notebooks']

Low confidence flag counts:
low_confidence_flag
False    4672
True      328

ATS percentile — min/mean/max per domain:
                 min   mean     max
primary_domain                     
IT              0.05  50.02  100.00
Data Science    0.10  50.05   99.40
HR              0.40  50.10   99.90
Legal           0.38  50.13   96.22
Engineering     0.53  50.27   97.88
Management      0.72  50.36   99.28

Semantic percentile — min/mean/max per domain:
                 min   mean     max
primary_domain                     
IT              0.05  50.02   99.96
Data Science    0.15  50.05  100.00
HR              0.50  50.10   99.60
Legal           3.27  50.13   96.85


# Notebook 07: Summary

## Status
Complete. All artifacts exported and validated.

## Artifacts Produced

| Artifact | Rows | Columns | Size |
|---|---|---|---|
| outputs/ats_benchmark_scores.csv | 5000 | 20 | 664 KB |
| outputs/benchmark_metadata.json | — | — | 3.4 KB |

## Methodology

Domain-stratified percentile ranks computed using scipy.stats.percentileofscore
with kind='rank'. Three scores receive independent domain percentile ranks:
ATS score, semantic display score, and domain coverage percentage. Gap count
excluded as directionally redundant with coverage. Experience band retained
as a display label only.

## Decisions Made

Stratification unit is domain-level only. Cross-domain percentiles rejected
due to confirmed seniority differences (Management mean experience 10.92 years
versus Data Science 5.54 years) and vocabulary alignment differences in semantic
scores (domain means ranging from 11.58 to 74.82).

Management (139 records) and Engineering (189 records) flagged
low_confidence_flag = True at the 200-record threshold. Percentile ranks are
computed for these domains but the Streamlit application must render a visible
reliability caveat alongside the percentile display.

Semantic display score must not be surfaced as an absolute quality signal.
Domain means differ by 63 points due to vocabulary alignment differences in the
synthetic corpus, not candidate quality differences. The domain percentile rank
is the only cross-candidate comparable output for this score.

Coverage percentile is compressed for HR (floor 40.82) and Management
(floor 19.42) due to narrow raw score ranges driven by 5-6 token candidate
profiles against 10-22 token domain pools. Use as a weak relative signal only.

## Key Findings

The three percentile signals are genuinely independent and will diverge for
individual candidates. A candidate can rank at the 23rd percentile on ATS score
and the 90th percentile on domain coverage simultaneously. This is correct
behaviour — ATS readiness, semantic match, and skill coverage measure different
dimensions of candidate fit. The Streamlit application must present them as
separate signals with separate labels, not aggregate them.

## Known Limitations

All scores are derived from synthetic candidate profiles with uniform templated
text fields. ATS completeness component carries zero variance across all 5000
candidates. Semantic scores are compressed due to skill sentence length asymmetry
against full job description text. Coverage percentiles reflect vocabulary-limited
extraction at a mean of 2.81 skills per job description versus 5.91 per candidate
profile.

All rescaling parameters, gap frequency tables, and confidence thresholds must
be recalibrated on real resume data before production use.

## Decisions Deferred

- Streamlit UI presentation logic for low_confidence_flag
- Recalibration of similarity rescaling parameters on real resume data
- Recalibration of gap frequency tables on real resume data
- Threshold decisions for strong versus weak percentile presentation in the UI

## Next Step

Streamlit application development. Begin with application architecture
planning before writing any UI code.